## Couche Bronze : Décès

L'objectif de cette couche est de stocker les données brutes au format Delta Lake dans RustFS, sans modifier les informations d'origine.

Deux colonnes techniques sont ajoutées :

_source : indique le fichier d'origine.

_ingere_le : indique la date et l'heure de l'ingestion.

### 1 - Créer le bucket RustFS

Le bucket est un space de stockage où nous allons enregistrer nos données. C'est pratique pour un visiuel sur ce que l'on fais.\
Disponible ici: http://localhost:9001 (identifiant dans le .env)

In [1]:
import sys
sys.path.append("/home/jovyan/work")

from pyspark.sql import functions as F
from utils.spark_utils import creer_session, RAW, LAKE, creer_bucket, format_tables

creer_bucket("datalake")

Bucket « datalake » déjà présent.


### 2 - Démarrer une session Spark

In [2]:
spark = creer_session("bronze_deces", lac=True, delta=True, postgres=False)

FMT = format_tables(spark)
print("Format utilisé :", FMT)

Préparation de Spark 4.2.1 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040
Format utilisé : delta


### 3 - Enregistrer les données -> bronze

In [3]:
deces_raw = (spark.read.parquet("/home/jovyan/work/data/deces.parquet")
            .withColumn("_source", F.lit("deces.parquet"))
            .withColumn("_ingere_le", F.current_timestamp()))

(deces_raw.write.format(FMT).mode("overwrite")
        .save(f"{LAKE}/bronze/deces"))

print("bronze/deces :",spark.read.format(FMT).load(f"{LAKE}/bronze/deces").count(), "lignes")

bronze/deces : 29411388 lignes


### Vérification

- Les colonnes d'origine sont toujours présentes.
- Les colonnes _source et _ingere_le ont bien été ajoutées.

In [ ]:
# lecture des données que l'on a stocker
bronze = spark.read.format(FMT).load(f"{LAKE}/bronze/deces")

print("Nombre de lignes :", bronze.count())

# shema des données
bronze.printSchema()

Nombre de lignes : 29411388
root
 |-- nom: string (nullable = true)
 |-- prenoms: string (nullable = true)
 |-- sexe: string (nullable = true)
 |-- date_naissance: string (nullable = true)
 |-- code_insee_naissance: string (nullable = true)
 |-- commune_naissance: string (nullable = true)
 |-- pays_naissance: string (nullable = true)
 |-- date_deces: string (nullable = true)
 |-- code_insee_deces: string (nullable = true)
 |-- numero_acte_deces: string (nullable = true)
 |-- fichier_origine: string (nullable = true)
 |-- opposition: boolean (nullable = true)
 |-- _source: string (nullable = true)
 |-- _ingere_le: timestamp (nullable = true)

